# Triple collocation: compare three products without a reference

This lab simulates three products that observe the same hidden signal. We know the hidden signal only because we created it. In a real study, the true signal is unavailable; classical triple collocation estimates each product's random-error variance from the three observed products.

The method relies on assumptions about linearity, signal-error orthogonality, mutually uncorrelated errors and stable statistics. This notebook demonstrates the calculation, not proof that those assumptions hold.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / 'implementations/python/triple-collocation').is_dir()), None)
if root is None:
    raise RuntimeError('Run this notebook from a repository checkout.')
sys.path.insert(0, str(root / 'implementations/python/triple-collocation'))
from triple_collocation import triple_collocation

rng = np.random.default_rng(2026)
print('Repository:', root)

## 1. Create a hidden signal and three products

Each product has its own offset, gain and independent random error. The error standard deviations are known here because we generate the data.

In [ ]:
n = 5_000
truth = rng.normal(loc=0.25, scale=0.12, size=n)
noise_sd = np.array([0.04, 0.07, 0.05])
products = np.column_stack([
    0.3 + 1.0 * truth + rng.normal(0, noise_sd[0], n),
    -0.1 + 1.4 * truth + rng.normal(0, noise_sd[1], n),
    0.8 + 0.7 * truth + rng.normal(0, noise_sd[2], n),
])
for i in range(3):
    print(f'Product {i+1}: mean={products[:, i].mean():.3f}, sd={products[:, i].std(ddof=1):.3f}')

## 2. Estimate the error variances

For product 1, the classical estimator is

$$\\widehat{\\sigma}_{e_1}^{2}=s_1^2-\\frac{s_{12}s_{13}}{s_{23}}.$$

The other two estimates follow by cycling the product indices. Each estimate is expressed in that product's units squared.

In [ ]:
result = triple_collocation(*products.T)
estimated = np.asarray(result.error_variances)
actual = noise_sd ** 2
print('Matched observations:', result.n_observations)
print('Estimated error variances:', estimated)
print('Known simulated error variances:', actual)
print('Estimated / known:', estimated / actual)

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(1, 4)
ax.bar(x - 0.18, actual, width=0.36, label='Known in simulation')
ax.bar(x + 0.18, estimated, width=0.36, label='Triple-collocation estimate')
ax.set_xticks(x, ['Product 1', 'Product 2', 'Product 3'])
ax.set_ylabel('Error variance (product units squared)')
ax.set_title('Estimated versus known error variance')
ax.legend()
plt.show()

## 3. Inspect the products

Pairwise scatterplots help reveal range differences, nonlinear patterns and unusual observations. They cannot establish error independence, but they are useful diagnostics.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
pairs = [(0, 1), (0, 2), (1, 2)]
for ax, (i, j) in zip(axes, pairs):
    ax.scatter(products[::5, i], products[::5, j], s=8, alpha=0.35)
    ax.set_xlabel(f'Product {i+1}')
    ax.set_ylabel(f'Product {j+1}')
    ax.set_title(f'Products {i+1} and {j+1}')
plt.tight_layout()
plt.show()

## 4. Missing observations: why one common sample matters

The implementation removes a row if any of the three products is non-finite. This keeps every covariance on the same sample. Pairwise deletion would make the covariance terms refer to different populations and invalidate the simple algebra.

In [ ]:
with_missing = products.copy()
with_missing[10, 0] = np.nan
with_missing[20, 1] = np.inf
complete = triple_collocation(*with_missing.T)
print('Input rows:', len(with_missing))
print('Rows retained for all three covariance estimates:', complete.n_observations)

## 5. What if two products share errors?

Products can share retrieval algorithms, training data, assimilation inputs or calibration sources. Such dependencies can correlate their errors and bias the classical estimates. A real validation study must investigate product lineage; a plausible-looking numerical result is not enough.

## Investigate

1. Change the sample size. How does the estimate vary?
2. Increase one product's noise. Does its estimate change as expected?
3. Add a shared noise component to two products and compare results across several random seeds.
4. Split the sample into wet/dry or seasonal regimes. Do the estimates remain stable?
5. Explain why a negative estimate should be reported as a diagnostic rather than clipped to zero.

Continue with the [method lesson](../methods/triple-collocation.md), [implementation walkthrough](../implementations/python/triple-collocation/implementation-walkthrough.md) and [exercise](../exercises/triple-collocation.md).